# 聖尼古拉斯與三姐妹：在 Colab 渲染

這本筆記本會把 [FelixF048/Santa](https://github.com/FelixF048/Santa) 的影片程式逐格渲染成 5 分鐘、1080p 的 mp4。

- 畫面是程式碼畫的（Canvas 2D），不使用影片生成模型。繪圖主要吃 CPU；有 GPU 時，影片編碼會自動改用 NVENC。
- 依序執行每一格即可。第 4 格是完整渲染，7200 格，視機器而定約需十幾到數十分鐘。

## 1. 取得程式

In [ ]:
import os
%cd /content
if os.path.isdir('Santa'):
    !git -C Santa pull --ff-only
else:
    !git clone https://github.com/FelixF048/Santa.git
%cd /content/Santa
!nvidia-smi -L || echo "沒有偵測到 GPU（仍可渲染，只是編碼用 CPU）"

## 2. 安裝：中文字型、ffmpeg、無頭瀏覽器

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk ffmpeg > /dev/null
!fc-cache -f
!node -v && ffmpeg -version | head -1
!npm install --no-audit --no-fund
!npx playwright install --with-deps chromium

## 3. 先看幾格單張畫面
確認字型與畫面正常，再做完整渲染。

In [ ]:
!node render/render.mjs --stills 13,56,136,238,249,281
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('out/stills/t_*.jpg')):
    print(f); display(Image(f, width=720))

## 4. 完整渲染
`--workers` 是平行分段數，預設取 CPU 核心數（上限 4）。輸出 `out/santa_silent.mp4`（無聲、字幕已燒進畫面）。

In [ ]:
!node render/render.mjs --fps 24
!ls -lh out/*.mp4

## 5. 聲音：配樂、音效、配音
用 edge-tts 合成旁白與台詞，輸出 `out/santa.mp4`。

**注意**：這一步會把字幕文字送到微軟的線上語音服務。不想外送文字就跳過，改用自己的錄音。配樂由 `colab/music.py` 用 numpy 合成。

In [ ]:
!pip -q install edge-tts
!python colab/music.py
!python colab/tts.py

## 6. 預覽與下載

In [ ]:
import os
from IPython.display import Video
final = 'out/santa.mp4' if os.path.exists('out/santa.mp4') else 'out/santa_silent.mp4'
print(final)
Video(final, embed=False, width=960)

In [ ]:
from google.colab import files
files.download(final)